In [166]:
N_DAYS = 1
csv_filename = f"results/results_jan_26_last_{N_DAYS}_days.csv"

In [309]:
from datetime import datetime, timedelta
import pandas as pd
import wandb


def flatten_dict(d, parent_key="", sep="/"):
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(flatten_dict(v, new_key, sep=sep).items())
        else:
            items.append((new_key, v))
    return dict(items)


def prepare_data(data):
    flattened_data = [flatten_dict(item) for item in data]
    return pd.DataFrame(flattened_data)


api = wandb.Api()

# Specify the project and entity (replace with your details)
project_path = "openproblems-comp/fast-tbg"

# Define N days
end_date = datetime.utcnow()
start_date = end_date - timedelta(days=N_DAYS)

# Convert dates to ISO format for filtering
start_iso = start_date.isoformat() + "Z"
end_iso = end_date.isoformat() + "Z"

# Query runs
runs = api.runs(
    project_path,
    filters={"created_at": {"$gte": start_iso, "$lte": end_iso}},
)

summary_list, config_list, name_list = [], [], []
for run in runs:
    # .summary contains the output keys/values for metrics like accuracy.
    #  We call ._json_dict to omit large files
    summary_list.append(run.summary._json_dict)
    # .config contains the hyperparameters.
    #  We remove special values that start with _.
    config_list.append({k: v for k, v in run.config.items() if not k.startswith("_")})
    # .name is the human-readable name of the run.
    name_list.append(run.name)
df_summary = prepare_data(summary_list)
df_config = prepare_data(config_list)
# df_name = prepare_data(name_list)

df = pd.concat([pd.DataFrame(name_list, columns=["name"]), df_summary, df_config], axis=1)
# runs_df = pd.DataFrame({"summary": summary_list, "config": config_list, "name": name_list})

print(f"Number of runs from the past {N_DAYS} days: {len(df)}")

KeyboardInterrupt: 

In [168]:
df.to_csv(csv_filename)

In [169]:
[s for s in df.keys() if "samples" in s]

['test/generated_samples/_type',
 'test/generated_samples/count',
 'test/generated_samples/filenames',
 'test/generated_samples/format',
 'test/generated_samples/height',
 'test/generated_samples/width',
 'test/num_eval_samples',
 'test/filtered_samples',
 'test/jarzynski/num_eval_samples',
 'test/jarzynski/samples_per_second',
 'test/jarzynski/samples_walltime',
 'test/samples_per_second',
 'test/samples_walltime',
 'val/generated_samples/_type',
 'val/generated_samples/count',
 'val/generated_samples/filenames',
 'val/generated_samples/format',
 'val/generated_samples/height',
 'val/generated_samples/width',
 'val/num_eval_samples',
 'model/sampling_config/num_eval_samples',
 'model/sampling_config/save_samples_path',
 'model/sampling_config/num_proposal_samples',
 'model/sampling_config/num_jarzynski_samples',
 'model/sampling_config/num_test_proposal_samples']

In [170]:
df["model/net/_target_"].value_counts()

model/net/_target_
src.models.components.tarflow.TarFlow                                          272
src.models.components.dit.DIT3D                                                 27
src.models.components.tbg.egnn_dynamics_ad2_cat_v2.EGNN_dynamics_AD2_cat_v2     25
src.models.components.tbg.egnn_dynamics_ad2_cat.EGNN_dynamics_AD2_cat           13
Name: count, dtype: int64

In [338]:
df = pd.read_csv(csv_filename)

In [339]:
import math

ESS_THRESHOLD = 0.0


def filterer(x):
    if isinstance(x, float) and not math.isfinite(x):
        return False
    return "table" in list(x)


print("Before filtering", df.shape)

tags_to_keep = ["jarz_final_v2", "jarz_final_v3"]

filtered_df = df.copy()

for index, row in filtered_df.iterrows():
    tag_found = False
    for tag in tags_to_keep:
        if tag in row["tags"]:
            tag_found = True
            break

    if not tag_found:
        filtered_df.drop(index, inplace=True)

for index, row in filtered_df.iterrows():
    if row["data/n_particles"] == 22 and "jarz_final_v2" in row["tags"]:
        filtered_df.drop(index, inplace=True)

# Sort the DataFrame
filtered_df = filtered_df.sort_values("data/n_particles")
print("After tag filter:", filtered_df.shape)

for index, row in filtered_df.iterrows():
    if row["data/n_particles"] == 22 and "jarz_final_v2" in row["tags"]:
        filtered_df.drop(index, inplace=True)

    if row["model/jarzynski_sampler/ess_threshold"] != ESS_THRESHOLD:
        filtered_df.drop(index, inplace=True)

print("After dropping rows:", filtered_df.shape)

# Select the desired columns
filtered_columns = [
    "model/net/_target_",
    "data/n_particles",
    "model/sampling_config/num_test_proposal_samples",
    "model/sampling_config/num_jarzynski_samples",
    "model/jarzynski_sampler/ess_threshold",
    "test/filtered_samples",
    "test/energy_w1",
    "test/effective_sample_size",
    "test/rama/torus_wasserstein",
    "test/resampled/energy_w1",
    "test/resampled/rama/torus_wasserstein",
    "test/jarzynski/energy_w1",
    "test/jarzynski/effective_sample_size",
    "test/jarzynski/rama/torus_wasserstein",
]
filtered_df = filtered_df[filtered_columns]

filtered_df.dropna(inplace=True)
print("After dropping nans:", filtered_df.shape)

Before filtering (337, 507)
After tag filter: (27, 507)
After dropping rows: (9, 507)
After dropping nans: (9, 14)


In [340]:
renamed_df = filtered_df.replace(
    {
        "src.models.components.tbg.egnn_dynamics_ad2_cat.EGNN_dynamics_AD2_cat": "EQ-CFM",
        "src.models.components.dit.DIT3D": "DiT-CFM",
        "src.models.components.tarflow.TarFlow": "TarFlow",
    }
).rename(
    columns={
        "model/net/_target_": "Model",
        "data/n_particles": "n_particles",
        "model/jarzynski_sampler/ess_threshold": "ess_threshold",
        "test/filtered_samples": "filtered_samples",
    }
)
# Remove 'model/sampling_config/' from column names
renamed_df.columns = renamed_df.columns.str.replace("model/sampling_config/", "", regex=False)

renamed_df["n_particles"] = renamed_df["n_particles"].astype(int)
renamed_df["num_test_proposal_samples"] = renamed_df["num_test_proposal_samples"].astype(int)
renamed_df["num_jarzynski_samples"] = renamed_df["num_jarzynski_samples"].astype(int)
renamed_df["filtered_samples"] = renamed_df["filtered_samples"].astype(int)

# List of columns to exclude from casting
exclude_columns = [
    "Model",
    "ess_threshold",
    "n_particles",
    "num_test_proposal_samples",
    "num_jarzynski_samples",
    "filtered_samples",
]

# Cast all columns except excluded ones to float
renamed_df = renamed_df.apply(
    lambda col: col.astype(float) if col.name not in exclude_columns else col
)

# Replace underscores with spaces in column names
renamed_df.columns = renamed_df.columns.str.replace("_", " ")
renamed_df.columns = renamed_df.columns.str.replace("test/", "")

renamed_df["ess threshold"] = renamed_df["ess threshold"].map(lambda x: f"{x:.1f}")

In [341]:
renamed_df.head()

,Model,n particles,num test proposal samples,num jarzynski samples,ess threshold,filtered samples,energy w1,effective sample size,rama/torus wasserstein,resampled/energy w1,resampled/rama/torus wasserstein,jarzynski/energy w1,jarzynski/effective sample size,jarzynski/rama/torus wasserstein
292,TarFlow,22,100000,100000,0.0,3550,5.565417e+07,0.000763,1.228845,1.070562,0.629940,0.812336,0.000502,0.753408
291,TarFlow,22,100000,100000,0.0,2619,1.511135e+11,0.000479,1.163175,2.608865,0.729702,56.815704,0.000010,2.806328
294,TarFlow,22,100000,100000,0.0,3044,8.616466e+04,0.000331,1.215304,1.437675,0.847275,11.718381,0.000010,2.821072
285,TarFlow,33,100000,100000,0.0,1201,1.148360e+08,0.000289,0.731233,1.238919,1.674858,1.604091,0.000640,1.492735
283,TarFlow,33,100000,100000,0.0,1485,9.181886e+11,0.000078,0.693612,4.519842,2.498477,3.008250,0.000051,2.633251


In [342]:
renamed_df.shape

(9, 14)

In [343]:
renamed_df.groupby(["Model", "n particles", "ess threshold"]).count()

num test proposal samples  \
Model   n particles ess threshold                              
TarFlow 22          0.0                                    3   
        33          0.0                                    3   
        42          0.0                                    3   

                                   num jarzynski samples  filtered samples  \
Model   n particles ess threshold                                            
TarFlow 22          0.0                                3                 3   
        33          0.0                                3                 3   
        42          0.0                                3                 3   

                                   energy w1  effective sample size  \
Model   n particles ess threshold                                     
TarFlow 22          0.0                    3                      3   
        33          0.0                    3                      3   
        42          0.0                    3                      3   

                                   rama/torus wasserstein  \
Model   n particles ess threshold                           
TarFlow 22          0.0                                 3   
        33          0.0                                 3   
        42          0.0                                 3   

                                   resampled/energy w1  \
Model   n particles ess threshold                        
TarFlow 22          0.0                              3   
        33          0.0                              3   
        42          0.0                              3   

                                   resampled/rama/torus wasserstein  \
Model   n particles ess threshold                                     
TarFlow 22          0.0                                           3   
        33          0.0                                           3   
        42          0.0                                           3   

                                   jarzynski/energy w1  \
Model   n particles ess threshold                        
TarFlow 22          0.0                              3   
        33          0.0                              3   
        42          0.0                              3   

                                   jarzynski/effective sample size  \
Model   n particles ess threshold                                    
TarFlow 22          0.0                                          3   
        33          0.0                                          3   
        42          0.0                                          3   

                                   jarzynski/rama/torus wasserstein  
Model   n particles ess threshold                                    
TarFlow 22          0.0                                           3  
        33          0.0                                           3  
        42          0.0                                           3

In [344]:
renamed_df.groupby(["Model", "n particles", "ess threshold"]).count()

# Group the DataFrame and count the occurrences
grouped_counts = renamed_df.groupby(["Model", "n particles", "ess threshold"]).count()

# Check if all counts are exactly 3
assert (grouped_counts == 3).all().all(), "Not all counts are equal to 3!"

print("All counts are exactly 3.")

All counts are exactly 3.


In [345]:
renamed_df.drop(
    [
        "num test proposal samples",
        "num jarzynski samples",
        "filtered samples",
        "ess threshold",
        "Model",
    ],
    axis=1,
    inplace=True,
)

In [346]:
mean_df = renamed_df.groupby(["n particles"]).mean()
std_df = renamed_df.groupby(["n particles"]).std()

formatted_df = mean_df.copy()  # Start with the same structure as mean_df

# Create a new DataFrame to store formatted rows
formatted_rows = []
for index, row in mean_df.iterrows():

    # Ensure index is a tuple (handles single- and multi-index cases)
    if not isinstance(index, tuple):
        index = (index,)

    formatted_row = {}
    for col in mean_df.columns:
        mean_value = row[col]
        std_value = std_df.loc[index, col]

        if col in ["energy w1"]:
            formatted_row[col] = f"{mean_value:.2f} ± {std_value:.2f}"  # Customize format here
        else:
            formatted_row[col] = f"{mean_value:.2f} ± {std_value:.2f}"  # Customize format here
    formatted_rows.append({**dict(zip(mean_df.index.names, index)), **formatted_row})

formatted_df = pd.DataFrame(formatted_rows)

# Create a MultiIndex for columns
formatted_df.columns = pd.MultiIndex.from_tuples(
    [
        ("", "n particles"),
        ("energy w1", "energy w1"),
        ("energy w1", "resampled/energy w1"),
        ("energy w1", "jarzynski/energy w1"),
        ("ess", "effective sample size"),
        ("ess", "jarzynsi/effective sample size"),
        ("torus wasserstein", "rama/torus wasserstein"),
        ("torus wasserstein", "resampled/rama/torus wasserstein"),
        ("torus wasserstein", "jarzynski/rama/torus wasserstein"),
    ]
)

# Rename subcolumns using a dictionary
formatted_df = formatted_df.rename(
    columns={
        "energy w1": "BG",
        "resampled/energy w1": "BG (R)",
        "jarzynski/energy w1": "ABG",
        "effective sample size": "BG",
        "jarzynsi/effective sample size": "ABG",
        "rama/torus wasserstein": "BG",
        "resampled/rama/torus wasserstein": "BG (R)",
        "jarzynski/rama/torus wasserstein": "ABG",
    },
    level=1,  # Only rename the subcolumn level
)

# Print LaTeX with MultiIndex columns
latex_table = formatted_df.to_latex(index=False, escape=False, multirow=True)
print(latex_table.replace(r"{r}", r"{c}"))

\begin{tabular}{rllllllll}
\toprule
 & \multicolumn{3}{c}{energy w1} & \multicolumn{2}{c}{ess} & \multicolumn{3}{c}{torus wasserstein} \\
n particles & BG & BG (R) & ABG & BG & ABG & BG & BG (R) & ABG \\
\midrule
22 & 50389757241.55 ± 87229351604.17 & 0.00 ± 0.00 & 1.20 ± 0.03 & 1.71 ± 0.80 & 0.74 ± 0.11 & 23.12 ± 29.69 & 0.00 ± 0.00 & 2.13 ± 1.19 \\
33 & 1367432806573.33 ± 1638791777953.98 & 0.00 ± 0.00 & 0.70 ± 0.03 & 2.78 ± 1.65 & 1.88 ± 0.54 & 2.08 ± 0.80 & 0.00 ± 0.00 & 1.81 ± 0.72 \\
42 & 6542464560607.88 ± 11331865358949.59 & 0.00 ± 0.00 & 1.76 ± 0.00 & 4.91 ± 0.86 & 3.40 ± 0.44 & 4.08 ± 1.75 & 0.00 ± 0.00 & 3.26 ± 0.74 \\
\bottomrule
\end{tabular}

